# FlowCast: Model Training
This notebook trains multiple machine learning models for traffic prediction.

## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

## 2. Load Processed Data

In [2]:
# Load the processed data
data = pd.read_csv('../data/processed/traffic_processed.csv')
print(f"Data shape: {data.shape}")
data.head()

Data shape: (2976, 11)


,Date,Day of the week,CarCount,BikeCount,BusCount,TruckCount,Total,Traffic Situation,hour,minute,AM/PM
0,10,2,31,0,4,4,39,0,0,0,0
1,10,2,49,0,3,3,55,0,0,15,0
2,10,2,46,0,3,6,55,0,0,30,0
3,10,2,51,0,2,5,58,0,0,45,0
4,10,2,57,6,15,16,94,1,1,0,0


## 3. Prepare Features and Target

In [3]:
# Define features and target
feature_columns = ['Date', 'Day of the week', 'CarCount', 'BikeCount', 'BusCount',
                   'TruckCount', 'Total', 'hour', 'minute', 'AM/PM']

X = data[feature_columns]
y = data['Traffic Situation'].values

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

Features shape: (2976, 10)
Target shape: (2976,)


## 4. Split Data

In [4]:
# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

Training set: (2380, 10)
Test set: (596, 10)


## 5. Feature Scaling

In [5]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed")

Feature scaling completed


## 6. Train Individual Models

In [6]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=0),
    'SVC': SVC(probability=True, random_state=0),
    'XGBoost': XGBClassifier(random_state=0),
    'AdaBoost': AdaBoostClassifier(random_state=0)
}

print("Models initialized")

Models initialized


In [7]:
# Train each model
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    print(f"{name} trained successfully\n")

print("All models trained!")

Training Logistic Regression...
Logistic Regression trained successfully

Training Random Forest...
Random Forest trained successfully

Training SVC...
SVC trained successfully

Training XGBoost...
XGBoost trained successfully

Training AdaBoost...
AdaBoost trained successfully

All models trained!


## 7. Train Voting Classifier

In [8]:
# Create voting classifier with all models
voting_clf = VotingClassifier(
    estimators=[
        ('lr', models['Logistic Regression']),
        ('rf', models['Random Forest']),
        ('svc', models['SVC']),
        ('xgb', models['XGBoost']),
        ('ada', models['AdaBoost'])
    ],
    voting='hard'
)

print("Training Voting Classifier...")
voting_clf.fit(X_train_scaled, y_train)
trained_models['Voting Classifier'] = voting_clf
print("Voting Classifier trained successfully!")

Training Voting Classifier...
Voting Classifier trained successfully!


## 8. Save Models and Scaler

In [9]:
import pickle

# Save the scaler
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save trained models
for name, model in trained_models.items():
    filename = name.lower().replace(' ', '_')
    with open(f'../models/{filename}.pkl', 'wb') as f:
        pickle.dump(model, f)

print("Models and scaler saved successfully!")

Models and scaler saved successfully!


## Summary
- Trained 5 individual models: Logistic Regression, Random Forest, SVC, XGBoost, AdaBoost
- Created and trained Voting Classifier ensemble
- All models saved for evaluation